CELL 1 — TITLE

### Task 2.2: Eye Aspect Ratio (EAR) Calculation  
**Course:** Advanced Data Science  
**Project:** Drowsiness Alert System  

**Description:**  
This task computes the Eye Aspect Ratio (EAR) using facial landmarks to determine whether the eyes are open or closed in real-time.

CELL 2 — OBJECTIVE

#### Objective
- Extract eye landmark points from MediaPipe
- Compute Eye Aspect Ratio (EAR)
- Display EAR value in real-time

CELL 3 — IMPORT LIBRARIES

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from scipy.spatial import distance

CELL 4 — INITIALIZE MEDIAPIPE

In [2]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh()

mp_draw = mp.solutions.drawing_utils

CELL 5 — EYE LANDMARK INDICES

In [3]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

CELL 6 — EAR FUNCTION

In [4]:
def calculate_EAR(eye_points):
    A = distance.euclidean(eye_points[1], eye_points[5])
    B = distance.euclidean(eye_points[2], eye_points[4])
    C = distance.euclidean(eye_points[0], eye_points[3])
    
    ear = (A + B) / (2.0 * C)
    return ear

CELL  — MAIN WEBCAM LOOP

In [5]:
cap = cv2.VideoCapture(0)

print("EAR Detection Running... Press 'q' to exit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            
            h, w, _ = frame.shape
            
            left_eye_points = []
            right_eye_points = []

            for idx in LEFT_EYE:
                x = int(face_landmarks.landmark[idx].x * w)
                y = int(face_landmarks.landmark[idx].y * h)
                left_eye_points.append((x, y))
                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            for idx in RIGHT_EYE:
                x = int(face_landmarks.landmark[idx].x * w)
                y = int(face_landmarks.landmark[idx].y * h)
                right_eye_points.append((x, y))
                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            left_eye_points = np.array(left_eye_points)
            right_eye_points = np.array(right_eye_points)

            left_EAR = calculate_EAR(left_eye_points)
            right_EAR = calculate_EAR(right_eye_points)

            EAR = (left_EAR + right_EAR) / 2.0

            cv2.putText(frame, f'EAR: {EAR:.2f}', (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("EAR Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

EAR Detection Running... Press 'q' to exit.


CELL 8 — RESULTS / OBSERVATION

#### Results / Observation

- EAR value is displayed in real-time
- When eyes are open, EAR is relatively high (~0.25–0.35)
- When blinking or closing eyes, EAR drops significantly

CELL 9 — CONCLUSION

#### Conclusion

The Eye Aspect Ratio (EAR) successfully provides a numerical measure of eye openness.  
This will be used in the next task to detect prolonged eye closure and trigger an alert.